# LLM Uncertainty Benchmark — Google Colab
Runs the full CP benchmark (5 models × 5 tasks, n=50) using Ollama on Colab.

**Runtime:** GPU is not required — Ollama runs on CPU. Use a standard Colab instance.

**Time estimate:** ~2–3 hours for all models × all tasks.

## 1. Install Ollama and start server

In [ ]:
import subprocess, time, os, requests, sys

# Step 1: install zstd
r = subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'],
                   capture_output=True, text=True)
print("zstd:", r.stdout[-200:] or r.stderr[-200:])

# Step 2: install Ollama
r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, capture_output=True, text=True)
print("ollama install:", r.stdout[-500:])
if r.returncode != 0:
    print("stderr:", r.stderr[-300:])

# Step 3: find binary
result = subprocess.run(
    ['find', '/usr', '/usr/local', '/opt', '/root', '-name', 'ollama', '-type', 'f'],
    capture_output=True, text=True
)
candidates = [p.strip() for p in result.stdout.splitlines() if p.strip()]
OLLAMA = candidates[0] if candidates else subprocess.run(
    ['which', 'ollama'], capture_output=True, text=True).stdout.strip() or None

if OLLAMA is None:
    raise RuntimeError("Ollama binary not found.")
print(f"Using: {OLLAMA}")

# Step 4: start server as a detached background process via nohup
os.system(f'nohup {OLLAMA} serve > /tmp/ollama.log 2>&1 &')

# Step 5: wait until ready
print("Waiting for Ollama server...", end='')
for i in range(40):
    try:
        if requests.get('http://localhost:11434', timeout=2).status_code == 200:
            print(f" ready after {i+1}s")
            break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    print("\nWARNING: server not responding — check /tmp/ollama.log")
    os.system('tail -20 /tmp/ollama.log')

## 2. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/SokhengDin/LLM-Uncertainty-Study.git
%cd LLM-Uncertainty-Study
!pip install -q numpy scikit-learn tqdm requests matplotlib

## 3. Pull models
Comment out models you don't need to save time.

In [ ]:
MODELS = [
    'qwen3.5:2b',
    'qwen3.5:4b',
    # 'qwen3.5:9b',     # ~6GB, slow on CPU
    'gemma4:e4b',
    'llama3.1:latest',
]

for model in MODELS:
    print(f'Pulling {model}...')
    ret = subprocess.run([OLLAMA, 'pull', model], capture_output=True, text=True)
    print(ret.stdout[-200:] if ret.stdout else 'done')

## 4. Configuration

In [ ]:
SAMPLES = 50      # n=50 → n_cal=27, quantile=0.926 (non-trivial CP)
PROMPT  = 'base'
ICL     = 'icl1'
ALPHA   = 0.1

DATASETS = [
    'mmlu_10k',
    'cosmosqa_10k',
    'hellaswag_10k',
    'halu_dialogue',
    'halu_summarization',
]

os.makedirs('outputs_base', exist_ok=True)
os.makedirs('figures', exist_ok=True)
print(f'Config: {len(MODELS)} models × {len(DATASETS)} tasks × n={SAMPLES}')

# Ensure Ollama server is running before generating logits
def ensure_ollama_running(ollama_bin):
    try:
        if requests.get('http://localhost:11434', timeout=2).status_code == 200:
            print("Ollama already running")
            return
    except Exception:
        pass

    print("Restarting Ollama server via nohup...")
    os.system(f'nohup {ollama_bin} serve > /tmp/ollama.log 2>&1 &')
    for i in range(40):
        try:
            if requests.get('http://localhost:11434', timeout=2).status_code == 200:
                print(f"Ollama ready after {i+1}s")
                return
        except Exception:
            time.sleep(1)
    os.system('tail -20 /tmp/ollama.log')
    raise RuntimeError("Ollama failed to start — see log above")

ensure_ollama_running(OLLAMA)

In [ ]:
import sys

for model in MODELS:
    for dataset in DATASETS:
        print(f'\n--- {model} | {dataset} ---')
        ret = subprocess.run([
            sys.executable, 'utils/generate_logits.py',
            '--model',         model,
            '--file',          f'{dataset}.json',
            '--prompt_method', PROMPT,
            '--few_shot',      '1',
            '--max_samples',   str(SAMPLES),
            '--output_dir',    'outputs_base',
        ], capture_output=True, text=True)
        print(ret.stdout[-300:] if ret.stdout else '')
        if ret.returncode != 0:
            print(f'ERROR: {ret.stderr[-500:]}')

In [ ]:
import sys

for model in MODELS:
    for dataset in DATASETS:
        print(f'\n--- {model} | {dataset} ---')
        ret = subprocess.run([
            sys.executable, 'utils/generate_logits.py',
            '--model',         model,
            '--file',          f'{dataset}.json',
            '--prompt_method', PROMPT,
            '--few_shot',      '1',
            '--max_samples',   str(SAMPLES),
            '--output_dir',    'outputs_base',
        ], capture_output=True, text=True)
        print(ret.stdout[-500:] if ret.stdout else '')
        if ret.returncode != 0:
            print(f'ERROR on {model} | {dataset}')
            print(ret.stderr[-1000:])
            break  # remove this line once errors are diagnosed
    else:
        continue
    break  # remove this line once errors are diagnosed

## 6. Run CP evaluation

In [ ]:
for model in MODELS:
    print(f'\n=== Evaluating {model} ===')
    ret = subprocess.run([
        sys.executable, 'main.py',
        '--model',          model,
        '--data_names',     *DATASETS,
        '--prompt_methods', PROMPT,
        '--icl_methods',    ICL,
        '--max_samples',    str(SAMPLES),
        '--alpha',          str(ALPHA),
    ], capture_output=False)
    if ret.returncode != 0:
        print(f'ERROR evaluating {model}')

## 7. Generate figures

In [ ]:
subprocess.run([
    sys.executable, 'plot_results.py',
    '--samples', str(SAMPLES),
    '--prompt',  PROMPT,
    '--icl',     ICL,
])

## 8. Display figures

In [ ]:
from IPython.display import Image, display
import glob

for png in sorted(glob.glob('figures/*.png')):
    print(f'\n{png}')
    display(Image(png))

## 9. Print summary table

In [ ]:
import json, numpy as np

key = f'{PROMPT}_{ICL}'
col = 16

header = f"{'Model':<20}" + ''.join(f"{d.split('_')[0]:>{col}}" for d in DATASETS)
print(header)
print(f"{'':20}" + ''.join(f"{'CR%/Acc%/SS':>{col}}" for _ in DATASETS))
print('-' * (20 + col * len(DATASETS)))

for model in MODELS:
    path = f'outputs_base/{model}_all_results.json'
    if not os.path.exists(path):
        print(f'{model:<20}  (no results)')
        continue
    res = json.load(open(path))
    row = f'{model:<20}'
    for d in DATASETS:
        if d not in res:
            row += f"{'N/A':>{col}}"
            continue
        acc = 100 * res[d]['Acc'][key]
        cr  = 100 * np.mean([res[d]['LAC_coverage'][key], res[d]['APS_coverage'][key]])
        ss  =       np.mean([res[d]['LAC_set_size'][key],  res[d]['APS_set_size'][key]])
        cell = f"{cr:.0f}/{acc:.0f}/{ss:.1f}"
        row += f"{cell:>{col}}"
    print(row)

## 10. Download results

In [ ]:
!zip -r benchmark_results.zip outputs_base/ figures/
from google.colab import files
files.download('benchmark_results.zip')